# 4. Inspect an HDF5 snapshot

RadHydropy writes solver snapshots in HDF5 format. This notebook runs a small Sod problem if necessary, opens one snapshot through the public loader, and inspects its metadata and typed fluid fields.

The important idea is that the HDF5 file stores the numerical state, while the loaded object restores units, mesh geometry, and runtime representation information.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO = Path('/content/RadHydropy')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/tkc004/RadHydropy.git', str(REPO)], check=True)
PINNED = ['numpy==2.5.3', 'scipy==1.18.1', 'matplotlib==3.10.9', 'h5py==3.15.1', 'PyYAML==6.0.3', 'unyt==3.0.4']
PINNED_MARKER = Path('/content/.radhydropy_dependencies_pinned')
if not PINNED_MARKER.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINNED, '-e', '.[docs]'], cwd=REPO, check=True)
    PINNED_MARKER.touch()
    print('Dependencies installed. Restarting the Colab runtime once to load the pinned NumPy/SciPy binaries.')
    os.kill(os.getpid(), 9)
sys.path.insert(0, str(REPO))
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from example import example_utils
import radhydropy.io as rio

In [ ]:
source_dir = REPO / 'example' / 'SodShock1D'
example_dir = Path('/content/radhydropy-sod-inspection')
if example_dir.exists():
    shutil.rmtree(example_dir)
example_dir.mkdir()
import yaml
config_data = yaml.safe_load((source_dir / 'sodshock1d.yaml').read_text())
config_data['par']['output']['directory'] = str(example_dir)
(example_dir / 'sodshock1d.yaml').write_text(yaml.safe_dump(config_data))
snapshot_file = example_dir / 'Output_002.hdf5'
if not snapshot_file.exists():
    from example.SodShock1D import sodshock1d
    sodshock1d.main(example_dir / 'sodshock1d.yaml')
config = example_utils.load_nested_example_config(example_dir / 'sodshock1d.yaml')
snapshot = rio.loadhdf5(config, snapshot_file)

print('Snapshot:', snapshot_file.name)
print('Coordinate system:', snapshot.par.simulation.coordinate_system)
print('Grid cells:', snapshot.par.mesh.grid_cells)
print('Fluid fields:', [name for name in dir(snapshot.fluid) if name.endswith('_radarray')])

In [ ]:
first = int(snapshot.par.mesh.ghost_cells)
count = int(snapshot.par.mesh.grid_cells)
last = first + count
units = snapshot.par.units.CodeUnits
boundary = np.asarray(snapshot.mesh.boundary_radarray.to_value(units.length_unit), dtype=float)
radius = 0.5 * (boundary[1:] + boundary[:-1])
density = np.asarray(snapshot.fluid.rho_radarray.to_value(units.density_unit), dtype=float)[first:last]
temperature = np.asarray(snapshot.fluid.temp_radarray.to_value(units.temperature_unit), dtype=float)[first:last]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(radius, density)
axes[0].set(xlabel='position [code length]', ylabel='density [code units]', title='Density')
axes[1].plot(radius, temperature)
axes[1].set(xlabel='position [code length]', ylabel='temperature [code units]', title='Temperature')
for axis in axes:
    axis.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## Try this

Change `Output_002.hdf5` to another snapshot and rerun the inspection cells. Compare how the density and temperature profiles evolve while the mesh and units remain part of the restored runtime state.

In [ ]:
# Optional: save the inspected snapshot to Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# save_dir = Path('/content/drive/MyDrive/RadHydropy/04_inspect_hdf5_snapshot')
# save_dir.mkdir(parents=True, exist_ok=True)
# for file in example_dir.glob('*'):
#     shutil.copy2(file, save_dir / file.name)
# print('Saved to:', save_dir)